# Clase 19 — Gradio + K-Means: Deploy y Segmentacion de Clientes

**Diplomado en Data Science Aplicada con Python**

---

**Objetivos de hoy:**
1. Desplegar un modelo supervisado con **Gradio** (interfaz web interactiva).
2. Entender que es el **aprendizaje no supervisado** y cuando usarlo.
3. Implementar **K-Means** paso a paso con el dataset **Mall Customers**.
4. **Perfilar** los clusters y darles nombres de negocio.

**Datasets:**
- Pima Indians Diabetes (seccion Gradio)
- Mall Customers (seccion K-Means)

---
## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

print("Imports OK")

---
## 1. Gradio — Desplegar el modelo de diabetes de la clase 18

En la clase pasada entrenamos un modelo para predecir diabetes. Ahora vamos a crear una **interfaz web interactiva** para que cualquier persona pueda usarlo sin tocar codigo.

### 1.1 Setup rapido: cargar diabetes y entrenar Random Forest

In [ ]:
from sklearn.datasets import fetch_openml

d = fetch_openml("diabetes", version=1, as_frame=True, parser="auto")
df_diab = d.frame.copy()
df_diab["target"] = (df_diab["class"] == "tested_positive").astype(int)
df_diab = df_diab.drop(columns=["class"])

for col in ["plas", "pres", "skin", "insu", "mass"]:
    df_diab[col] = df_diab[col].replace(0, np.nan).fillna(df_diab[col].median())

X_d = df_diab.drop(columns=["target"])
y_d = df_diab["target"]

X_tr, X_te, y_tr, y_te = train_test_split(
    X_d, y_d, test_size=0.2, random_state=42, stratify=y_d
)

rf_diab = RandomForestClassifier(
    n_estimators=200, max_depth=8, class_weight="balanced", random_state=42
)
rf_diab.fit(X_tr, y_tr)

print(f"AUC diabetes: {roc_auc_score(y_te, rf_diab.predict_proba(X_te)[:, 1]):.3f}")

### 1.5 Ejemplo 4: El modelo real — predictor de diabetes

In [ ]:
# !pip install gradio
import gradio as gr


def predecir_diabetes(embarazos, glucosa, presion, pliegue, insulina, bmi, pedigri, edad):
    paciente = pd.DataFrame([{
        "preg": embarazos, "plas": glucosa, "pres": presion,
        "skin": pliegue, "insu": insulina, "mass": bmi,
        "pedi": pedigri, "age": edad
    }])
    proba = rf_diab.predict_proba(paciente)[0, 1]
    nivel = "ALTO" if proba >= 0.5 else "MEDIO" if proba >= 0.2 else "BAJO"
    return f"Riesgo de Diabetes: {proba:.1%} ({nivel})"

### 1.2 Ejemplo 1: La funcion mas simple — un saludo

In [ ]:
def saludar(nombre):
    return f"Hola, {nombre}! Bienvenido al diplomado."

demo_saludo = gr.Interface(fn=saludar, inputs="text", outputs="text")
demo_saludo.launch()

**Interpretacion:** Gradio toma una funcion Python y genera una interfaz web. `inputs='text'` crea un campo de texto. `outputs='text'` muestra el resultado como texto. Tres lineas.

### 1.3 Ejemplo 2: Dos inputs numericos — multiplicacion

In [ ]:
def multiplicar(a, b):
    return a * b

demo_mult = gr.Interface(
    fn=multiplicar,
    inputs=[gr.Number(label="Número A"), gr.Number(label="Número B")],
    outputs=gr.Number(label="Resultado"))
demo_mult.launch()

**Interpretacion:** Cuando hay multiples inputs, se pasan como una lista. Cada elemento corresponde a un parametro de la funcion, en orden. `gr.Number` crea un campo numerico.

### 1.4 Ejemplo 3: Sliders y Dropdowns — calculadora de propina

In [ ]:
def calcular_propina(monto, porcentaje, dividir_entre):
    propina = monto * (porcentaje / 100)
    total = monto + propina
    por_persona = total / dividir_entre
    return f"Propina: ${propina:.2f} | Total: ${total:.2f} | Por persona: ${por_persona:.2f}"

demo_propina = gr.Interface(
    fn=calcular_propina,
    inputs=[
        gr.Slider(10, 500, value=100, label="Monto de la cuenta ($)"),
        gr.Slider(5, 30, value=15, step=1, label="Propina (%)"),
        gr.Dropdown([1, 2, 3, 4, 5, 6], value=2, label="Dividir entre"),
    ],
    outputs="text",
    title="Calculadora de Propina")
demo_propina.launch()

**Interpretacion:** `gr.Slider` limita el rango (el usuario no puede poner valores imposibles). `gr.Dropdown` ofrece opciones fijas. Gradio mezcla tipos de input sin problema.

### 1.6 Interfaz interactiva

In [ ]:
demo = gr.Interface(
    fn=predecir_diabetes,
    inputs=[
        gr.Slider(0, 17, value=1, step=1, label="Embarazos"),
        gr.Slider(0, 200, value=100, label="Glucosa (mg/dL)"),
        gr.Slider(0, 122, value=70, label="Presion arterial"),
        gr.Slider(0, 99, value=20, label="Pliegue cutaneo"),
        gr.Slider(0, 846, value=80, label="Insulina (uU/mL)"),
        gr.Slider(10, 67, value=25, step=0.5, label="BMI"),
        gr.Slider(0, 2.5, value=0.5, step=0.01, label="Pedigri"),
        gr.Slider(21, 81, value=30, step=1, label="Edad"),
    ],
    outputs=gr.Text(label="Resultado"),
    title="Predictor de Riesgo de Diabetes"
)
demo.launch()

**Interpretacion:** Gradio genera una interfaz web interactiva. Cada slider corresponde a una variable clinica. El modelo predice en tiempo real cuando movemos los sliders.

### 1.7 Ejemplo 5: Devolver gráficos — Scatter interactivo

Hasta ahora Gradio devolvía texto o números. Ahora vamos a devolver un **gráfico**.
El usuario elige dos variables con Dropdowns y Gradio genera un scatter de Plotly.

In [ ]:
# Cargar Mall Customers para los ejemplos de gráficos
df_mall = pd.read_csv("mall_customers.csv")
df_mall = df_mall.rename(columns={"Annual Income (k$)": "income", "Spending Score (1-100)": "spending"})

variables_disponibles = ["Age", "income", "spending"]

def scatter_interactivo(var_x, var_y):
    """Recibe dos nombres de variable, devuelve un scatter de Plotly."""
    fig = px.scatter(df_mall, x=var_x, y=var_y,
                     title=f"{var_x} vs {var_y}",
                     opacity=0.7, color_discrete_sequence=["#C82B40"])
    fig.update_layout(template="plotly_white")
    return fig

demo_scatter = gr.Interface(
    fn=scatter_interactivo,
    inputs=[
        gr.Dropdown(variables_disponibles, value="income", label="Eje X"),
        gr.Dropdown(variables_disponibles, value="spending", label="Eje Y"),
    ],
    outputs=gr.Plot(label="Gráfico"),
    title="Explorador de variables")
demo_scatter.launch()

**Interpretación:** `gr.Plot()` como output permite devolver cualquier figura de matplotlib o Plotly.
La función retorna el objeto `fig` directamente — Gradio se encarga de renderizarlo.
Ahora el usuario puede explorar relaciones entre variables sin escribir código.

### 1.8 Ejemplo 6: Múltiples outputs — scatter con color + texto

Podemos devolver **varios outputs a la vez**: un texto con la correlación y un gráfico coloreado por una tercera variable.

In [ ]:
def scatter_con_color(var_x, var_y, var_color):
    """Devuelve texto con correlación + scatter coloreado."""
    corr = df_mall[var_x].corr(df_mall[var_y])
    texto = f"Correlación {var_x} vs {var_y}: {corr:.3f}"
    
    fig = px.scatter(df_mall, x=var_x, y=var_y, color=var_color,
                     title=f"{var_x} vs {var_y} (color: {var_color})",
                     opacity=0.7,
                     color_discrete_map={"Male": "#2563EB", "Female": "#C82B40"})
    fig.update_layout(template="plotly_white")
    return texto, fig

demo_color = gr.Interface(
    fn=scatter_con_color,
    inputs=[
        gr.Dropdown(variables_disponibles, value="income", label="Eje X"),
        gr.Dropdown(variables_disponibles, value="spending", label="Eje Y"),
        gr.Dropdown(["Gender"], value="Gender", label="Color"),
    ],
    outputs=[gr.Text(label="Correlación"), gr.Plot(label="Gráfico")],
    title="Explorador con color")
demo_color.launch()

**Interpretación:** Para múltiples outputs, la función retorna una **tupla** `(texto, fig)` y en `outputs` pasamos una **lista** `[gr.Text(), gr.Plot()]`. Gradio muestra ambos lado a lado. Este patrón es fundamental: en producción casi siempre queremos devolver la predicción + una visualización que la explique.

### Ejercicio 1b — Calculadora estadística con Gradio

Crea una demo de Gradio que:
1. Reciba el nombre de una variable con `gr.Dropdown` (opciones: Age, income, spending)
2. Calcule: **media**, **desviación estándar** y **skewness** (`df[var].skew()`)
3. Devuelva:
   - Un **texto** con las 3 estadísticas formateadas
   - Un **gráfico** (`gr.Plot()`) con un `kdeplot` de seaborn mostrando la distribución de esa variable
4. Usa `outputs=[gr.Text(), gr.Plot()]` para devolver ambos

**Pista:** para devolver una figura de matplotlib/seaborn en Gradio:
```python
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(data=df_mall, x=variable, fill=True, ax=ax)
return texto, fig
```

In [ ]:
# Tu codigo aqui


### Ejercicio 1 — Probar el demo

Usa la funcion `predecir_diabetes()` para comparar:
- **Alto riesgo:** mujer de 45 anios, glucosa 180, BMI 35, 3 embarazos, presion 80, pliegue 30, insulina 150, pedigri 0.8
- **Bajo riesgo:** mujer de 25 anios, glucosa 85, BMI 22, 0 embarazos, presion 65, pliegue 15, insulina 50, pedigri 0.3

Tiene sentido la diferencia?

In [ ]:
# Tu codigo aqui


---
## 2. Aprendizaje No Supervisado — Introduccion

Hasta ahora siempre teniamos un **target** (y). Ahora vamos a trabajar con datos donde **NO hay respuesta correcta**. El objetivo: descubrir **grupos (clusters) ocultos** en los datos.

| Supervisado | No Supervisado |
|---|---|
| Tiene etiqueta (y) | No tiene etiqueta |
| Predecir un resultado | Descubrir estructura |
| Clasificacion, Regresion | Clustering, Reduccion de dimensionalidad |

---
## 3. Mall Customers — Cargar y explorar

In [ ]:
df = pd.read_csv("mall_customers.csv")
df = df.rename(columns={
    "Annual Income (k$)": "income",
    "Spending Score (1-100)": "spending"
})
print(df.shape)
df.head()

### 3.1 Analisis Exploratorio (EDA)

In [ ]:
df.describe()

**Interpretacion:** 200 clientes. Ingreso de 15k a 137k. Spending Score de 1 a 99. No hay valores faltantes.

In [ ]:
fig = px.scatter(
    df, x="income", y="spending", color="Gender",
    title="Ingreso vs Gasto por genero", opacity=0.7,
    labels={"income": "Ingreso anual (k$)", "spending": "Spending Score"},
    color_discrete_map={"Male": "#2563EB", "Female": "#C82B40"}
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Se intuyen grupos: hay clientes con alto ingreso y alto gasto (arriba derecha), alto ingreso y bajo gasto (abajo derecha), etc. Puede un algoritmo encontrar estos grupos automaticamente?

In [ ]:
fig = px.histogram(
    df, x="income", nbins=20,
    title="Distribucion del ingreso",
    color_discrete_sequence=["#C82B40"]
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** El ingreso anual se distribuye de forma relativamente uniforme entre 15k y 137k, con una ligera concentracion alrededor de 50k-80k.

In [ ]:
fig = px.histogram(
    df, x="spending", nbins=20,
    title="Distribucion del Spending Score",
    color_discrete_sequence=["#2563EB"]
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** El Spending Score tiene una distribucion con picos en los extremos y el centro, lo que sugiere la presencia de grupos distintos de comportamiento de gasto.

---
## 4. K-Means — El algoritmo

K-Means es el algoritmo de clustering mas popular. Funciona asi:

1. **Elegir K** — el numero de clusters que queremos.
2. **Centroides aleatorios** — se colocan K puntos al azar como centros iniciales.
3. **Asignar** — cada punto se asigna al centroide mas cercano.
4. **Recalcular** — se mueve cada centroide al promedio de sus puntos asignados.
5. **Repetir** — pasos 3 y 4 hasta que los centroides no se muevan (convergencia).

### K-Means animado paso a paso

Antes de usar `KMeans` de sklearn, veamos **exactamente** qué hace el algoritmo por dentro. Vamos a programar las iteraciones a mano y crear una animación con Plotly donde se ve cómo los centroides se mueven y los puntos cambian de color en cada iteración.

In [ ]:
# K-Means animado: iteracion por iteracion
X_anim = df[["income", "spending"]].values
K = 5
COLORS = ["#2563EB", "#C82B40", "#16A34A", "#EA580C", "#7C3AED"]

# Inicializar centroides al azar
rng_anim = np.random.default_rng(42)
centroids = X_anim[rng_anim.choice(len(X_anim), K, replace=False)].copy()

# Guardar cada paso para la animacion
frames_data = []

for iteration in range(7):
    # Paso A: asignar cada punto al centroide mas cercano
    dists = np.array([np.linalg.norm(X_anim - c, axis=1) for c in centroids])
    labels = dists.argmin(axis=0)
    
    # Guardar puntos
    for i in range(len(X_anim)):
        frames_data.append({
            "income": X_anim[i, 0], "spending": X_anim[i, 1],
            "cluster": f"Cluster {labels[i]}",
            "tipo": "Cliente", "size": 6,
            "iteracion": f"Iteracion {iteration}"
        })
    # Guardar centroides
    for k_idx in range(K):
        frames_data.append({
            "income": centroids[k_idx, 0], "spending": centroids[k_idx, 1],
            "cluster": f"Cluster {k_idx}",
            "tipo": "CENTROIDE", "size": 20,
            "iteracion": f"Iteracion {iteration}"
        })
    
    # Paso B: recalcular centroides
    for k_idx in range(K):
        mask = labels == k_idx
        if mask.sum() > 0:
            centroids[k_idx] = X_anim[mask].mean(axis=0)

df_anim = pd.DataFrame(frames_data)

# Animacion con Plotly
fig = px.scatter(
    df_anim, x="income", y="spending",
    color="cluster", symbol="tipo",
    symbol_map={"Cliente": "circle", "CENTROIDE": "x"},
    size="size", size_max=18,
    animation_frame="iteracion",
    title="K-Means paso a paso: los centroides se mueven hasta estabilizarse",
    labels={"income": "Ingreso anual (k$)", "spending": "Spending Score"},
    color_discrete_sequence=COLORS,
    opacity=0.8)

fig.update_layout(
    template="plotly_white", width=800, height=550,
    legend_title_text="")
fig.show()

**Interpretacion:** Usen el boton de play o arrastren el slider de iteraciones. Observen:
- **Iteracion 0:** los centroides (X grandes) estan en posiciones aleatorias. Los colores son arbitrarios.
- **Iteraciones 1-3:** los centroides se mueven rapidamente hacia el centro de sus grupos.
- **Iteraciones 4-6:** los centroides apenas se mueven — el algoritmo **convergio**.

Esto es exactamente lo que  de sklearn hace internamente. La unica diferencia es que sklearn lo repite 10 veces () con inicios distintos y se queda con el mejor resultado.

### 4.1 Metodo del Codo (Elbow Method)

Como elegimos K? Probamos varios valores y graficamos la **inercia** (suma de distancias al centroide). Buscamos el "codo" donde agregar mas clusters ya no ayuda mucho.

In [ ]:
X_km = df[["income", "spending"]].values

inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_km)
    inertias.append(km.inertia_)

fig = px.line(
    x=list(K_range), y=inertias, markers=True,
    title="Metodo del Codo — Mall Customers",
    labels={"x": "K (clusters)", "y": "Inercia"}
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** El codo esta en **K=5**. Despues de 5 clusters, agregar mas no reduce significativamente la inercia.

### 4.2 Ajustar K-Means con K=5

In [ ]:
km = KMeans(n_clusters=5, random_state=42, n_init=10)
df["cluster"] = km.fit_predict(X_km)
print(f"Clusters: {df['cluster'].value_counts().sort_index().to_dict()}")

### 4.3 Visualizar clusters

In [ ]:
fig = px.scatter(
    df, x="income", y="spending",
    color=df["cluster"].astype(str),
    title="5 Segmentos de Clientes — K-Means",
    labels={"income": "Ingreso anual (k$)", "spending": "Spending Score", "color": "Segmento"},
    color_discrete_sequence=["#2563EB", "#C82B40", "#16A34A", "#EA580C", "#7C3AED"],
    opacity=0.8
)

# Agregar centroides
for i, c in enumerate(km.cluster_centers_):
    fig.add_trace(go.Scatter(
        x=[c[0]], y=[c[1]], mode="markers",
        marker=dict(
            size=15, symbol="x",
            color=["#2563EB", "#C82B40", "#16A34A", "#EA580C", "#7C3AED"][i],
            line=dict(width=2, color="black")
        ),
        name=f"Centroide {i}", showlegend=False
    ))

fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los 5 segmentos son claros: K-Means encontro exactamente los grupos que intuiamos en el scatter original. Los centroides (X) son el "centro de gravedad" de cada grupo.

### 4.4 Silhouette Score — ¿Qué tan buenos son los clusters?

El codo nos dice cuántos clusters usar, pero no qué tan **bien separados** están. El Silhouette Score mide eso:
- **+1**: el punto está muy dentro de su cluster (perfecto)
- **0**: está en la frontera entre dos clusters
- **-1**: probablemente está en el cluster equivocado

Promedio > 0.5 = bueno. > 0.7 = excelente.

In [ ]:
from sklearn.metrics import silhouette_score

sil_scores = []
for k in range(2, 11):
    km_sil = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_sil = km_sil.fit_predict(X_km)
    sil = silhouette_score(X_km, labels_sil)
    sil_scores.append({"K": k, "Silhouette": sil})
    
df_sil = pd.DataFrame(sil_scores)
fig = px.line(df_sil, x="K", y="Silhouette", markers=True,
    title="Silhouette Score por K",
    labels={"Silhouette": "Silhouette Score"})
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** El Silhouette Score mas alto confirma que K=5 es una buena eleccion. Los clusters estan bien separados entre si. Usar tanto el codo como el Silhouette da mas confianza en la decision.

---
## 5. Interpretar los clusters — Profiling

### 5.1 Promedio por cluster

In [ ]:
perfil = df.groupby("cluster")[["Age", "income", "spending"]].mean().round(1)
perfil.columns = ["Edad promedio", "Ingreso promedio (k$)", "Spending Score promedio"]
print(perfil)

**Interpretacion:** Cada cluster tiene un perfil distinto. Podemos asignar nombres de negocio segun la combinacion de ingreso y gasto.

### 5.2 Nombres de negocio para cada segmento

In [ ]:
nombres = {}
for c in range(5):
    inc = perfil.loc[c, "Ingreso promedio (k$)"]
    sp = perfil.loc[c, "Spending Score promedio"]
    if inc > 70 and sp > 60:
        nombres[c] = "Premium"
    elif inc > 70 and sp < 40:
        nombres[c] = "Ahorradores ricos"
    elif inc < 40 and sp > 60:
        nombres[c] = "Gastadores"
    elif inc < 40 and sp < 40:
        nombres[c] = "Cautelosos"
    else:
        nombres[c] = "Promedio"

df["perfil"] = df["cluster"].map(nombres)
print(df["perfil"].value_counts())

### 5.3 Grafica de barras por perfil

In [ ]:
perfil_plot = df.groupby("perfil")[["income", "spending"]].mean().round(1)
fig = px.bar(
    perfil_plot.reset_index(), x="perfil", y=["income", "spending"],
    barmode="group", title="Perfil promedio por segmento",
    color_discrete_sequence=["#2563EB", "#C82B40"]
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Los "Premium" tienen ingreso y gasto altos. Los "Ahorradores ricos" tienen ingreso alto pero gastan poco — son la **oportunidad de negocio mas grande**.

### 5.4 Distribucion de genero por segmento

In [ ]:
gender_cluster = df.groupby(["perfil", "Gender"]).size().reset_index(name="count")
fig = px.bar(
    gender_cluster, x="perfil", y="count", color="Gender",
    title="Genero por segmento", barmode="group",
    color_discrete_map={"Male": "#2563EB", "Female": "#C82B40"}
)
fig.update_layout(template="plotly_white")
fig.show()

**Interpretacion:** Algunos segmentos pueden tener composicion de genero distinta. Esto puede informar las campanias de marketing.

---
### Ejercicio 2 — Segmentacion completa desde cero

Realiza el proceso completo de segmentacion:
1. Metodo del codo (elbow) para elegir K
2. Ajustar K-Means
3. Calcular el perfil (promedio) de cada cluster
4. Asignar nombres de negocio
5. Visualizar con scatter
6. Proponer una **estrategia de marketing** para cada segmento

In [ ]:
# Tu codigo aqui


---
### Ejercicio 3 — Agregar edad como tercera variable

1. Usa `income`, `spending` y `Age` como variables.
2. **IMPORTANTE:** Escala las variables con `StandardScaler` (porque Age esta en otra escala).
3. Aplica K-Means y encuentra el K optimo con el metodo del codo.
4. Crea un **scatter 3D** con `px.scatter_3d`.
5. Cambian los segmentos respecto a usar solo 2 variables?

In [ ]:
# Tu codigo aqui


---
## Resumen de la clase

| Tema | Concepto clave |
|---|---|
| **Gradio** | Interfaz web interactiva para modelos ML con sliders |
| **Aprendizaje No Supervisado** | No hay etiqueta (y); se busca estructura oculta |
| **K-Means** | Algoritmo de clustering: asigna puntos al centroide mas cercano |
| **Metodo del Codo** | Elegir K graficando inercia vs numero de clusters |
| **Silhouette Score** | Mide que tan bien separados estan los clusters (+1 perfecto, -1 mal asignado) |
| **Profiling** | Interpretar clusters con promedios y nombres de negocio |
| **StandardScaler** | Necesario cuando las variables tienen escalas distintas |